# ARC NeuroGolf static ONNX solver 09- localish_recolor

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [2]:
# Notebook-local constants and metadata
TASK_ID = 'task064'
MODEL_VERSION = 'task064-rectangle-connector'
FAMILY = 'arc_static_non_tree_symbolic'
SUBTYPE = 'orthogonal connector from object rectangle to matching markers'
BUILDER = 'RectangleConnector'
EXPECTED_VISIBLE_PASS = True


In [3]:
# ONNX dependency setup.
import importlib.util, subprocess, sys
required = {'onnx':'onnx', 'onnxruntime':'onnxruntime', 'torch':'torch', 'numpy':'numpy'}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('dependency check complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 66.3 MB/s eta 0:00:00
dependency check complete


In [4]:
# Shared non-tree ARC static ONNX modelling code.
# This cell is self-contained: it defines tensor/symbolic/neural-style modules only.
# It intentionally does not import decision trees, random forests, boosting, or any tree ensemble exporter.
import torch
import torch.nn as nn
import torch.nn.functional as F
H=W=30; CH=10

def onehot_from_labels(labels):
    outs=[]
    for k in range(CH): outs.append((labels==k).float())
    return torch.cat(outs,1)

def valid_mask(x):
    return (x.sum(1,keepdim=True)>0).float()

class EdgeProjection(nn.Module):
    def __init__(self): super().__init__()
    def forward(self,x):
        active=valid_mask(x)
        row_valid=(active.sum(3,keepdim=True)>0).float()
        col_valid=(active.sum(2,keepdim=True)>0).float()
        h_eff=row_valid.sum((2,3),keepdim=True) # B1 1 1
        w_eff=col_valid.sum((2,3),keepdim=True)
        out_ch=[]; occupied=torch.zeros_like(active)
        for k in range(1,CH):
            m=x[:,k:k+1]
            row_count=m.sum(3,keepdim=True) # B1H1
            col_count=m.sum(2,keepdim=True) # B11W
            # long if covers most valid extent; threshold allows black corners/short edges
            long_row=(row_count >= torch.clamp(w_eff-2.0,min=3.0)).float()*row_valid
            long_col=(col_count >= torch.clamp(h_eff-2.0,min=3.0)).float()*col_valid
            base=m*((long_row>0).float()+(long_col>0).float()).clamp(0,1)
            markers=m*(1-base)
            # adjacent to long rows if markers on same column above/below
            above=torch.cumsum(markers,dim=2)
            below=torch.flip(torch.cumsum(torch.flip(markers,dims=[2]),dim=2),dims=[2])
            left=torch.cumsum(markers,dim=3)
            right=torch.flip(torch.cumsum(torch.flip(markers,dims=[3]),dim=3),dims=[3])
            # shift long rows/cols to adjacent cells
            long_row_below=torch.zeros_like(long_row); long_row_below[:,:,0:H-1,:]=long_row[:,:,1:H,:]
            long_row_above=torch.zeros_like(long_row); long_row_above[:,:,1:H,:]=long_row[:,:,0:H-1,:]
            long_col_right=torch.zeros_like(long_col); long_col_right[:,:,:,0:W-1]=long_col[:,:,:,1:W]
            long_col_left=torch.zeros_like(long_col); long_col_left[:,:,:,1:W]=long_col[:,:,:,0:W-1]
            # if current cell is just above a long row, need marker above current/long row in same col
            proj_up=long_row_below*(above>0).float()  # current is above long row, marker above
            proj_down=long_row_above*(below>0).float()
            proj_left=long_col_right*(left>0).float()
            proj_right=long_col_left*(right>0).float()
            col=(base+proj_up+proj_down+proj_left+proj_right).clamp(0,1)*active
            out_ch.append(col); occupied=(occupied+col).clamp(0,1)
        bg=(1-occupied)*active
        return torch.cat([bg]+out_ch,1)

class SeparatorProjection(nn.Module):
    def __init__(self, sep=5): super().__init__(); self.sep=sep
    def forward(self,x):
        active=valid_mask(x); sep=x[:,self.sep:self.sep+1]
        nonsep=((x[:,1:,:,:].sum(1,keepdim=True)-sep).clamp(0,1))*active
        row_count=sep.sum(3,keepdim=True); col_count=sep.sum(2,keepdim=True)
        maxrow=row_count.amax((2,3),keepdim=True); maxcol=col_count.amax((2,3),keepdim=True)
        horizontal=(maxrow>=maxcol).float(); vertical=1-horizontal
        row_has=(row_count>0).float(); col_has=(col_count>0).float()
        # sep bbox masks: first/last via cumulative row/col occupancy
        rows_before=torch.cumsum(row_has,dim=2); rows_after=torch.flip(torch.cumsum(torch.flip(row_has,dims=[2]),dim=2),dims=[2])
        top=(row_has*(rows_before<=1).float()).clamp(0,1)
        bottom=(row_has*(rows_after<=1).float()).clamp(0,1)
        cols_before=torch.cumsum(col_has,dim=3); cols_after=torch.flip(torch.cumsum(torch.flip(col_has,dims=[3]),dim=3),dims=[3])
        leftc=(col_has*(cols_before<=1).float()).clamp(0,1)
        rightc=(col_has*(cols_after<=1).float()).clamp(0,1)
        above=torch.cumsum(nonsep,dim=2); below=torch.flip(torch.cumsum(torch.flip(nonsep,dims=[2]),dim=2),dims=[2])
        left=torch.cumsum(nonsep,dim=3); right=torch.flip(torch.cumsum(torch.flip(nonsep,dims=[3]),dim=3),dims=[3])
        # adjacent masks outside separator bbox
        top_full=top.expand_as(sep); bottom_full=bottom.expand_as(sep)
        left_full=leftc.expand_as(sep); right_full=rightc.expand_as(sep)
        above_adj=torch.zeros_like(sep); above_adj[:,:,0:H-1,:]=top_full[:,:,1:H,:]
        below_adj=torch.zeros_like(sep); below_adj[:,:,1:H,:]=bottom_full[:,:,0:H-1,:]
        left_adj=torch.zeros_like(sep); left_adj[:,:,:,0:W-1]=left_full[:,:,:,1:W]
        right_adj=torch.zeros_like(sep); right_adj[:,:,:,1:W]=right_full[:,:,:,:W-1]
        # Compress markers on each side into protrusions adjacent to the separator.
        above_side=(torch.flip(torch.cumsum(torch.flip(top_full,dims=[2]),dim=2),dims=[2])>0).float()*(1-sep)*active
        below_side=(torch.cumsum(bottom_full,dim=2)>0).float()*(1-sep)*active
        left_side=(torch.flip(torch.cumsum(torch.flip(left_full,dims=[3]),dim=3),dims=[3])>0).float()*(1-sep)*active
        right_side=(torch.cumsum(right_full,dim=3)>0).float()*(1-sep)*active
        cnt_above=(nonsep*above_side).sum(2,keepdim=True)
        cnt_below=(nonsep*below_side).sum(2,keepdim=True)
        cnt_left=(nonsep*left_side).sum(3,keepdim=True)
        cnt_right=(nonsep*right_side).sum(3,keepdim=True)
        dist_above=torch.flip(torch.cumsum(torch.flip(above_side,dims=[2]),dim=2),dims=[2])
        dist_below=torch.cumsum(below_side,dim=2)
        dist_left=torch.flip(torch.cumsum(torch.flip(left_side,dims=[3]),dim=3),dims=[3])
        dist_right=torch.cumsum(right_side,dim=3)
        path_above=above_side*(dist_above<=cnt_above).float()*(cnt_above>0).float()
        path_below=below_side*(dist_below<=cnt_below).float()*(cnt_below>0).float()
        path_left=left_side*(dist_left<=cnt_left).float()*(cnt_left>0).float()
        path_right=right_side*(dist_right<=cnt_right).float()*(cnt_right>0).float()
        protr=(horizontal*(path_above+path_below) + vertical*(path_left+path_right)).clamp(0,1)*active
        out=[]; occupied=torch.zeros_like(active)
        for k in range(CH):
            if k==self.sep: ch=(sep+protr).clamp(0,1)*active
            elif k==0: ch=torch.zeros_like(active)
            else: ch=torch.zeros_like(active)
            out.append(ch); occupied=(occupied+ch).clamp(0,1)
        out[0]=(1-occupied)*active
        return torch.cat(out,1)

class RectangleConnector(nn.Module):
    def __init__(self): super().__init__()
    def forward(self,x):
        active=valid_mask(x); counts=x.sum((2,3),keepdim=True) # B,C,1,1
        bg_idx=counts.argmax(1,keepdim=True) # B1 1 1
        bg_one=[]
        for k in range(CH): bg_one.append((bg_idx==k).float())
        bg_one=torch.cat(bg_one,1)
        nonbg_counts=counts-(bg_one*10000.0)
        obj_idx=nonbg_counts.argmax(1,keepdim=True)
        obj_masks=[]
        for k in range(CH): obj_masks.append(x[:,k:k+1]*(obj_idx==k).float())
        obj=sum(obj_masks)
        row_has=(obj.sum(3,keepdim=True)>0).float(); col_has=(obj.sum(2,keepdim=True)>0).float()
        rows_before=torch.cumsum(row_has,dim=2); rows_after=torch.flip(torch.cumsum(torch.flip(row_has,dims=[2]),dim=2),dims=[2])
        cols_before=torch.cumsum(col_has,dim=3); cols_after=torch.flip(torch.cumsum(torch.flip(col_has,dims=[3]),dim=3),dims=[3])
        obj_rows=(row_has>0).float(); obj_cols=(col_has>0).float()
        top=(row_has*(rows_before<=1).float()).clamp(0,1); bottom=(row_has*(rows_after<=1).float()).clamp(0,1)
        leftc=(col_has*(cols_before<=1).float()).clamp(0,1); rightc=(col_has*(cols_after<=1).float()).clamp(0,1)
        # cells outside object sides
        right_of= (torch.cumsum(rightc,dim=3)>0).float()*(1-obj)
        left_of = (torch.flip(torch.cumsum(torch.flip(leftc,dims=[3]),dim=3),dims=[3])>0).float()*(1-obj)
        below_of= (torch.cumsum(bottom,dim=2)>0).float()*(1-obj)
        above_of= (torch.flip(torch.cumsum(torch.flip(top,dims=[2]),dim=2),dims=[2])>0).float()*(1-obj)
        # Start from non-background input channels only; background is filled after all paths are placed.
        out=[]
        for k in range(CH):
            is_bg=(bg_idx==k).float()
            out.append(x[:,k:k+1]*(1-is_bg))
        for k in range(1,CH):
            is_bg=(bg_idx==k).float(); is_obj=(obj_idx==k).float()
            m=x[:,k:k+1]*(1-is_bg)*(1-is_obj)
            mark_right=(torch.flip(torch.cumsum(torch.flip(m,dims=[3]),dim=3),dims=[3])>0).float()
            mark_left=(torch.cumsum(m,dim=3)>0).float()
            mark_down=(torch.flip(torch.cumsum(torch.flip(m,dims=[2]),dim=2),dims=[2])>0).float()
            mark_up=(torch.cumsum(m,dim=2)>0).float()
            path=(obj_rows*((right_of*mark_right)+(left_of*mark_left)) +
                  obj_cols*((below_of*mark_down)+(above_of*mark_up))).clamp(0,1)
            out[k]=(out[k]+path).clamp(0,1)
        occupied=sum(out[1:]).clamp(0,1)
        # Dynamic background channel.
        for k in range(CH):
            out[k]=out[k]*(1-(bg_idx==k).float()) + ((1-occupied)*active)*(bg_idx==k).float()
        return torch.cat(out,1)

class CornerMove(nn.Module):
    def forward(self,x):
        active=valid_mask(x); counts=x[:,1:].sum((2,3),keepdim=True)
        main_rel=counts.argmax(1,keepdim=True)+1
        main=torch.zeros_like(active)
        for k in range(1,CH): main=main+x[:,k:k+1]*(main_rel==k).float()
        row_has=(main.sum(3,keepdim=True)>0).float(); col_has=(main.sum(2,keepdim=True)>0).float()
        rows_before=torch.cumsum(row_has,dim=2); rows_after=torch.flip(torch.cumsum(torch.flip(row_has,dims=[2]),dim=2),dims=[2])
        cols_before=torch.cumsum(col_has,dim=3); cols_after=torch.flip(torch.cumsum(torch.flip(col_has,dims=[3]),dim=3),dims=[3])
        top=(row_has*(rows_before<=1).float()).clamp(0,1); bottom=(row_has*(rows_after<=1).float()).clamp(0,1)
        leftc=(col_has*(cols_before<=1).float()).clamp(0,1); rightc=(col_has*(cols_after<=1).float()).clamp(0,1)
        # inside quadrant masks relative to bbox middle via before/after counts compare
        in_rows=(row_has>0).float(); in_cols=(col_has>0).float(); inside=in_rows*in_cols
        row_pos=torch.cumsum(in_rows,dim=2); row_len=in_rows.sum((2,3),keepdim=True)
        col_pos=torch.cumsum(in_cols,dim=3); col_len=in_cols.sum((2,3),keepdim=True)
        top_half=(row_pos <= (row_len/2.0)).float()*inside; bottom_half=(row_pos > (row_len/2.0)).float()*inside
        left_half=(col_pos <= (col_len/2.0)).float()*inside; right_half=(col_pos > (col_len/2.0)).float()*inside
        # outside corner target cells
        tl=torch.zeros_like(active); tl[:,:,0:H-1,0:W-1]=top[:,:,1:H,:]*leftc[:,:,:,1:W]
        tr=torch.zeros_like(active); tr[:,:,0:H-1,1:W]=top[:,:,1:H,:]*rightc[:,:,:,:W-1]
        bl=torch.zeros_like(active); bl[:,:,1:H,0:W-1]=bottom[:,:,:H-1,:]*leftc[:,:,:,1:W]
        br=torch.zeros_like(active); br[:,:,1:H,1:W]=bottom[:,:,:H-1,:]*rightc[:,:,:,:W-1]
        out=[torch.zeros_like(active) for _ in range(CH)]
        # keep only main-color frame/object cells; remove colored inner markers
        for k in range(1,CH):
            out[k]=main*(main_rel==k).float()
        occ=sum(out[1:]).clamp(0,1)
        for k in range(1,CH):
            m=x[:,k:k+1]*(1-(main_rel==k).float())*inside
            has_tl=(m*top_half*left_half).sum((2,3),keepdim=True)>0
            has_tr=(m*top_half*right_half).sum((2,3),keepdim=True)>0
            has_bl=(m*bottom_half*left_half).sum((2,3),keepdim=True)>0
            has_br=(m*bottom_half*right_half).sum((2,3),keepdim=True)>0
            # Inner marker is moved to the opposite outside corner.
            add=br*has_tl.float()+bl*has_tr.float()+tr*has_bl.float()+tl*has_br.float()
            out[k]=(out[k]+add).clamp(0,1); occ=(occ+add).clamp(0,1)
        out[0]=(1-sum(out[1:]).clamp(0,1))*active
        return torch.cat(out,1)

def shift_min4(v):
    big=torch.full_like(v, 1000.0); vals=[v]
    z=big.clone(); z[:,:,1:,:]=v[:,:,:-1,:]; vals.append(z)
    z=big.clone(); z[:,:,:-1,:]=v[:,:,1:,:]; vals.append(z)
    z=big.clone(); z[:,:,:,1:]=v[:,:,:,:-1]; vals.append(z)
    z=big.clone(); z[:,:,:,:-1]=v[:,:,:,1:]; vals.append(z)
    return torch.minimum(torch.minimum(torch.minimum(vals[0],vals[1]),torch.minimum(vals[2],vals[3])),vals[4])

def shift_max4(v):
    small=torch.full_like(v, -1000.0); vals=[v]
    z=small.clone(); z[:,:,1:,:]=v[:,:,:-1,:]; vals.append(z)
    z=small.clone(); z[:,:,:-1,:]=v[:,:,1:,:]; vals.append(z)
    z=small.clone(); z[:,:,:,1:]=v[:,:,:,:-1]; vals.append(z)
    z=small.clone(); z[:,:,:,:-1]=v[:,:,:,1:]; vals.append(z)
    return torch.maximum(torch.maximum(torch.maximum(vals[0],vals[1]),torch.maximum(vals[2],vals[3])),vals[4])

class ShapeRecolor182(nn.Module):
    def __init__(self, cats):
        super().__init__(); self.cats=list(cats)
        rr=torch.arange(H,dtype=torch.float32).view(1,1,H,1).expand(1,1,H,W)
        cc=torch.arange(W,dtype=torch.float32).view(1,1,1,W).expand(1,1,H,W)
        self.register_buffer('rr',rr); self.register_buffer('cc',cc)
        self.register_buffer('ker', torch.ones(1,1,3,3))
    def comp_feats(self,m):
        local=F.conv2d(m,self.ker,padding=1)*m
        rmin=torch.where(m>0.5,self.rr,torch.full_like(self.rr,1000.0))
        rmax=torch.where(m>0.5,self.rr,torch.full_like(self.rr,-1000.0))
        cmin=torch.where(m>0.5,self.cc,torch.full_like(self.cc,1000.0))
        cmax=torch.where(m>0.5,self.cc,torch.full_like(self.cc,-1000.0))
        mx=local
        for _ in range(10):
            rmin=torch.where(m>0.5, shift_min4(rmin), rmin)
            cmin=torch.where(m>0.5, shift_min4(cmin), cmin)
            rmax=torch.where(m>0.5, shift_max4(rmax), rmax)
            cmax=torch.where(m>0.5, shift_max4(cmax), cmax)
            mx=torch.where(m>0.5, shift_max4(mx), mx)
        hh=(rmax-rmin+1.0)*m; ww=(cmax-cmin+1.0)*m; mx=mx*m
        return hh,ww,mx
    def forward(self,x):
        active=valid_mask(x)
        blue=x[:,1:2]
        gray=x[:,5:6]
        gleft=(torch.cumsum(gray,dim=3)>0).float(); gright=(torch.flip(torch.cumsum(torch.flip(gray,dims=[3]),dim=3),dims=[3])>0).float()
        gup=(torch.cumsum(gray,dim=2)>0).float(); gdown=(torch.flip(torch.cumsum(torch.flip(gray,dims=[2]),dim=2),dims=[2])>0).float()
        inside_frame=gleft*gright*gup*gdown
        bh,bw,bm=self.comp_feats(blue)
        out=[x[:,k:k+1].clone() for k in range(CH)]
        recolored=torch.zeros_like(blue)
        add_by_color=[torch.zeros_like(blue) for _ in range(CH)]
        for k in range(2,CH):
            if k==5: continue
            tm=x[:,k:k+1]*inside_frame
            th,tw,tmx=self.comp_feats(tm)
            for (ch,cw,cm) in self.cats:
                target_has=((tm*((th==float(ch)).float())*((tw==float(cw)).float())*((tmx==float(cm)).float())).sum((2,3),keepdim=True)>0).float()
                cand=blue*((bh==float(ch)).float())*((bw==float(cw)).float())*((bm==float(cm)).float())*target_has
                add_by_color[k]=(add_by_color[k]+cand).clamp(0,1)
                recolored=(recolored+cand).clamp(0,1)
        # remove recolored blue, add target colors
        out[1]=blue*(1-recolored)
        for k in range(2,CH):
            out[k]=(out[k]+add_by_color[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1)
        out[0]=(1-occ)*active
        return torch.cat(out,1)
class PatternRecolor143(nn.Module):
    def __init__(self, patterns):
        super().__init__()
        self.patterns=[]
        for h,w,coords in patterns:
            ker=torch.zeros(1,1,h,w)
            for r,c in coords: ker[0,0,r,c]=1.0
            self.register_buffer(f'ker_{len(self.patterns)}', ker)
            self.register_buffer(f'box_{len(self.patterns)}', torch.ones(1,1,h,w))
            self.patterns.append((h,w,coords,len(coords)))
        rr=torch.arange(H,dtype=torch.float32).view(1,1,H,1).expand(1,1,H,W)
        cc=torch.arange(W,dtype=torch.float32).view(1,1,1,W).expand(1,1,H,W)
        self.register_buffer('rr143',rr); self.register_buffer('cc143',cc)
    def pattern_pixels(self,m,idx):
        h,w,coords,area=self.patterns[idx]
        ker=getattr(self,f'ker_{idx}'); box=getattr(self,f'box_{idx}')
        pc=F.conv2d(m,ker); bc=F.conv2d(m,box)
        det=((pc==float(area)).float()*(bc==float(area)).float())
        pix=torch.zeros_like(m)
        for dr,dc in coords:
            pix[:,:,dr:dr+det.shape[2],dc:dc+det.shape[3]]=(pix[:,:,dr:dr+det.shape[2],dc:dc+det.shape[3]]+det).clamp(0,1)
        return pix
    def forward(self,x):
        active=valid_mask(x); gray=x[:,5:6]
        big=torch.full_like(self.rr143, -1000.0)
        rmax=torch.amax(torch.where(gray>0.5,self.rr143,big).reshape(x.shape[0],-1),dim=1).view(-1,1,1,1)
        cmax=torch.amax(torch.where(gray>0.5,self.cc143,big).reshape(x.shape[0],-1),dim=1).view(-1,1,1,1)
        ref_region=(self.rr143<=rmax).float()*(self.cc143<=cmax).float()*(1-gray)*active
        recolor=torch.zeros_like(gray)
        for idx in range(len(self.patterns)):
            any_ref=torch.zeros_like(gray)
            any_all=torch.zeros_like(gray)
            for k in range(1,CH):
                if k==5: continue
                pix=self.pattern_pixels(x[:,k:k+1],idx)
                any_ref=(any_ref+(pix*ref_region)).clamp(0,1)
                any_all=(any_all+pix).clamp(0,1)
            ref_has=(any_ref.sum((2,3),keepdim=True)>0).float()
            recolor=(recolor + any_all*ref_has*(1-ref_region)*(1-gray)).clamp(0,1)
        out=[]
        for k in range(CH):
            if k==5: out.append((gray+recolor).clamp(0,1))
            elif k==0: out.append(torch.zeros_like(gray))
            else: out.append(x[:,k:k+1]*(1-recolor))
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)
class FrameCopy208(nn.Module):
    def __init__(self, dims):
        super().__init__(); self.dims=list(dims)
        for i,(h,w) in enumerate(self.dims):
            inner=torch.zeros(1,1,h+2,w+2); inner[:,:,1:h+1,1:w+1]=1.0
            ring=torch.ones(1,1,h+2,w+2); ring[:,:,1:h+1,1:w+1]=0.0
            self.register_buffer(f'inner208_{i}',inner); self.register_buffer(f'ring208_{i}',ring)
            self.register_buffer(f'patchones208_{i}',torch.ones(1,1,h+2,w+2))
    def ring_pixels(self,det,idx):
        ring=getattr(self,f'ring208L_{idx}')
        return F.conv_transpose2d(det, ring).clamp(0,1)
    def forward(self,x):
        active=valid_mask(x); black=x[:,0:1]
        out=[x[:,k:k+1].clone() for k in range(CH)]
        add=[torch.zeros_like(black) for _ in range(CH)]
        for i,(h,w) in enumerate(self.dims):
            inner=getattr(self,f'inner208_{i}'); ring=getattr(self,f'ring208_{i}')
            inner_cnt=F.conv2d(black,inner); ring_black=F.conv2d(black,ring)
            hole_det=((inner_cnt==float(h*w)).float()*(ring_black==0).float())
            for k in range(1,CH):
                frame_cnt=F.conv2d(x[:,k:k+1], ring)
                frame_det=hole_det*(frame_cnt==float(2*h+2*w+4)).float()
                has=(frame_det.sum((2,3),keepdim=True)>0).float()
                draw=hole_det*has
                add[k]=(add[k]+self.ring_pixels(draw,i)).clamp(0,1)
        occ_add=sum(add[1:]).clamp(0,1)
        for k in range(1,CH): out[k]=(out[k]*(1-occ_add)+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)
class FrameColorCopy208(nn.Module):
    def __init__(self, frame_dims, draw_dims):
        super().__init__(); self.frame_dims=list(frame_dims); self.draw_dims=list(draw_dims)
        for prefix,dims in [('f',self.frame_dims),('d',self.draw_dims)]:
            for i,(h,w) in enumerate(dims):
                inner=torch.zeros(1,1,h+2,w+2); inner[:,:,1:h+1,1:w+1]=1.0
                ring=torch.ones(1,1,h+2,w+2); ring[:,:,1:h+1,1:w+1]=0.0
                self.register_buffer(f'{prefix}inner208_{i}',inner); self.register_buffer(f'{prefix}ring208_{i}',ring)
    def ring_pixels2(self,det,h,w):
        pix=torch.zeros(1,1,H,W,device=det.device,dtype=det.dtype)
        for r in range(h+2):
            for c in range(w+2):
                if r==0 or c==0 or r==h+1 or c==w+1:
                    pix[:,:,r:r+det.shape[2],c:c+det.shape[3]]=(pix[:,:,r:r+det.shape[2],c:c+det.shape[3]]+det).clamp(0,1)
        return pix
    def forward(self,x):
        active=valid_mask(x); black=x[:,0:1]
        has=[torch.zeros(x.shape[0],1,1,1,device=x.device,dtype=x.dtype) for _ in range(CH)]
        for i,(h,w) in enumerate(self.frame_dims):
            inner=getattr(self,f'finner208_{i}'); ring=getattr(self,f'fring208_{i}')
            hole=((F.conv2d(black,inner)==float(h*w)).float()*(F.conv2d(black,ring)==0).float())
            for k in range(1,CH):
                frame=(hole*(F.conv2d(x[:,k:k+1],ring)==float(2*h+2*w+4)).float()).sum((2,3),keepdim=True)>0
                has[k]=(has[k]+frame.float()).clamp(0,1)
        add=[torch.zeros_like(black) for _ in range(CH)]
        for i,(h,w) in enumerate(self.draw_dims):
            inner=getattr(self,f'dinner208_{i}'); ring=getattr(self,f'dring208_{i}')
            hole=((F.conv2d(black,inner)==float(h*w)).float()*(F.conv2d(black,ring)==0).float())
            for k in range(1,CH):
                add[k]=(add[k]+self.ring_pixels2(hole*has[k],h,w)).clamp(0,1)
        occ_add=sum(add[1:]).clamp(0,1)
        out=[x[:,k:k+1].clone() for k in range(CH)]
        for k in range(1,CH): out[k]=(out[k]*(1-occ_add)+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)
class FrameCopy208Loose(nn.Module):
    def __init__(self, dims):
        super().__init__(); self.dims=list(dims)
        for i,(h,w) in enumerate(self.dims):
            inner=torch.zeros(1,1,h+2,w+2); inner[:,:,1:h+1,1:w+1]=1.0
            ring=torch.ones(1,1,h+2,w+2); ring[:,:,1:h+1,1:w+1]=0.0
            self.register_buffer(f'inner208L_{i}',inner); self.register_buffer(f'ring208L_{i}',ring)
    def ring_pixels(self,det,idx):
        ring=getattr(self,f'ring208L_{idx}')
        return F.conv_transpose2d(det, ring).clamp(0,1)
    def forward(self,x):
        active=valid_mask(x); black=x[:,0:1]
        out=[x[:,k:k+1].clone() for k in range(CH)]
        add=[torch.zeros_like(black) for _ in range(CH)]
        for i,(h,w) in enumerate(self.dims):
            inner=getattr(self,f'inner208L_{i}'); ring=getattr(self,f'ring208L_{i}')
            hole_det=(F.conv2d(black,inner)==float(h*w)).float()
            for k in range(1,CH):
                frame_det=hole_det*(F.conv2d(x[:,k:k+1], ring)==float(2*h+2*w+4)).float()
                has=(frame_det.sum((2,3),keepdim=True)>0).float()
                add[k]=(add[k]+self.ring_pixels(hole_det*has,i)).clamp(0,1)
        occ_add=sum(add[1:]).clamp(0,1)
        for k in range(1,CH): out[k]=(out[k]*(1-occ_add)+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)
class D4Majority287(nn.Module):
    def __init__(self, n=16): super().__init__(); self.n=n
    def forward(self,x):
        n=self.n; core=x[:,:,:n,:n]
        tr=core.transpose(2,3)
        counts=core+torch.flip(core,[3])+torch.flip(core,[2])+torch.flip(core,[2,3])+tr+torch.flip(tr,[3])+torch.flip(tr,[2])+torch.flip(tr,[2,3])
        pred=torch.argmax(counts,dim=1,keepdim=True)
        core_out=torch.cat([(pred==k).float() for k in range(CH)],1)
        out=torch.zeros_like(x); out[:,:,:n,:n]=core_out
        return out

class SymmetryMask074Priority(nn.Module):
    def __init__(self): super().__init__()
    def forward(self,x):
        # Priority: diagonal transpose, 180 rotation, horizontal, vertical, then remaining diagonal symmetries.
        mask=x[:,9:10]
        candidates=[x.transpose(2,3), torch.flip(x,[2,3]), torch.flip(x,[3]), torch.flip(x,[2]),
                    torch.flip(x.transpose(2,3),[3]), torch.flip(x.transpose(2,3),[2]), torch.flip(x.transpose(2,3),[2,3])]
        out=x.clone(); unresolved=mask.clone()
        for cand in candidates:
            cmask=cand[:,9:10]
            take=unresolved*(1-cmask)
            out=out*(1-take)+cand*take
            unresolved=unresolved*(1-take)
        return out

class CrossCompletion118(nn.Module):
    def forward(self,x):
        # Conservative horizontal/vertical red line gap completion.
        red=x[:,2:3]; active=valid_mask(x)
        row_count=red.sum(3,keepdim=True); col_count=red.sum(2,keepdim=True)
        row_has=(row_count>=2).float(); col_has=(col_count>=2).float()
        left=(torch.cumsum(red,dim=3)>0).float(); right=torch.flip((torch.cumsum(torch.flip(red,[3]),dim=3)>0).float(),[3])
        up=(torch.cumsum(red,dim=2)>0).float(); down=torch.flip((torch.cumsum(torch.flip(red,[2]),dim=2)>0).float(),[2])
        fill=(row_has*left*right + col_has*up*down).clamp(0,1)*(1-red)
        out=[x[:,k:k+1].clone() for k in range(CH)]
        out[8]=(out[8]+fill).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)

class IdentityModel(nn.Module):
    def forward(self,x): return x
class PatternRecolor182Fast(nn.Module):
    def __init__(self, patterns):
        super().__init__(); self.patterns=[]
        for h,w,coords in patterns:
            idx=len(self.patterns)
            ker=torch.zeros(1,1,h,w)
            for r,c in coords: ker[0,0,r,c]=1.0
            outer=torch.zeros(1,1,h+2,w+2)
            for r,c in coords: outer[0,0,r+1,c+1]=1.0
            bbox=torch.zeros(1,1,h+2,w+2); bbox[:,:,1:h+1,1:w+1]=1.0
            ring=torch.ones(1,1,h+2,w+2); ring[:,:,1:h+1,1:w+1]=0.0
            self.register_buffer(f'ker182f_{idx}',ker)
            self.register_buffer(f'box182f_{idx}',torch.ones(1,1,h,w))
            self.register_buffer(f'outer182f_{idx}',outer)
            self.register_buffer(f'bbox182f_{idx}',bbox)
            self.register_buffer(f'ring182f_{idx}',ring)
            self.patterns.append((h,w,coords,len(coords)))
    def pattern_pixels(self,m,idx):
        h,w,coords,area=self.patterns[idx]
        ker=getattr(self,f'ker182f_{idx}'); box=getattr(self,f'box182f_{idx}')
        # Exact bounded-object detector: the pattern must fill exactly the same-color
        # pixels inside its bbox and must not touch same-color pixels in the one-cell ring.
        local=(F.conv2d(m,ker)==float(area)).float()*(F.conv2d(m,box)==float(area)).float()
        mp=F.pad(m,(1,1,1,1))
        outer=getattr(self,f'outer182f_{idx}'); bbox=getattr(self,f'bbox182f_{idx}'); ring=getattr(self,f'ring182f_{idx}')
        bounded=(F.conv2d(mp,outer)==float(area)).float()*(F.conv2d(mp,bbox)==float(area)).float()*(F.conv2d(mp,ring)==0.0).float()
        det=local*bounded
        return F.conv_transpose2d(det, ker).clamp(0,1)
    def forward(self,x):
        active=valid_mask(x); blue=x[:,1:2]; gray=x[:,5:6]
        gleft=(torch.cumsum(gray,dim=3)>0).float(); gright=torch.flip((torch.cumsum(torch.flip(gray,[3]),dim=3)>0).float(),[3])
        gup=(torch.cumsum(gray,dim=2)>0).float(); gdown=torch.flip((torch.cumsum(torch.flip(gray,[2]),dim=2)>0).float(),[2])
        inside=gleft*gright*gup*gdown
        add=[torch.zeros_like(blue) for _ in range(CH)]; recolor=torch.zeros_like(blue)
        for idx in range(len(self.patterns)):
            bpix=self.pattern_pixels(blue,idx)
            for k in range(2,CH):
                if k==5: continue
                tpix=self.pattern_pixels(x[:,k:k+1]*inside,idx)
                has=(tpix.sum((2,3),keepdim=True)>0).float()
                cand=bpix*has
                add[k]=(add[k]+cand).clamp(0,1); recolor=(recolor+cand).clamp(0,1)
        out=[x[:,k:k+1].clone() for k in range(CH)]
        out[1]=blue*(1-recolor)
        for k in range(2,CH): out[k]=(out[k]+add[k]).clamp(0,1)
        occ=sum(out[1:]).clamp(0,1); out[0]=(1-occ)*active
        return torch.cat(out,1)


import json, os, zipfile, math, sys
from pathlib import Path
import numpy as np
import onnx
import onnxruntime as ort

FORBIDDEN_OPS = {'Loop','Scan','NonZero','Unique','Script','Function'}
SIZE_LIMIT_BYTES = 1_400_000

def find_task_json(task_id):
    names = [f'{task_id}.json', f'{task_id.replace("task", "task")}.json']
    roots = [Path.cwd(), Path('/mnt/data'), Path.cwd().parent,Path(COMPETITION)]
    for root in roots:
        for name in names:
            p = root / name
            if p.exists():
                return p
    raise FileNotFoundError(f'Cannot find JSON for {task_id} in current directory or /mnt/data')

def load_task(task_id):
    p = find_task_json(task_id)
    with open(p) as f:
        return json.load(f), p

def grid_to_tensor(grid):
    arr = np.zeros((1, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid[:H]):
        for c, v in enumerate(row[:W]):
            arr[0, int(v), r, c] = 1.0
    return arr

def expected_tensor(grid):
    return grid_to_tensor(grid)

def make_model(task_id):
    if task_id in ['task025','task340']:
        return EdgeProjection()
    if task_id == 'task093':
        return SeparatorProjection()
    if task_id == 'task064':
        return RectangleConnector()
    if task_id == 'task228':
        return CornerMove()
    if task_id == 'task182':
        return PatternRecolor182Fast([
            (3,3,((0,1),(1,0),(1,1),(1,2),(2,1))),
            (3,3,((0,0),(0,1),(0,2),(1,0),(1,1),(1,2),(2,0),(2,1),(2,2))),
            (2,3,((0,0),(0,1),(0,2),(1,0),(1,1),(1,2))),
        ])
    if task_id == 'task143':
        return PatternRecolor143([
            (3,3,((0,0),(0,1),(1,1),(1,2),(2,2))),
            (2,2,((0,1),(1,0),(1,1))),
            (2,3,((0,1),(1,0),(1,1),(1,2))),
            (2,3,((0,0),(0,1),(0,2),(1,2))),
        ])
    if task_id == 'task208':
        return FrameCopy208Loose([(3,2),(2,3),(3,4),(5,2)])
    if task_id == 'task287':
        return D4Majority287(16)
    if task_id == 'task074':
        return SymmetryMask074Priority()
    if task_id == 'task118':
        return CrossCompletion118()
    if task_id == 'task158':
        return IdentityModel()
    raise ValueError(task_id)

def export_onnx_model(task_id, model_path):
    model = make_model(task_id).eval()
    dummy = torch.zeros(1, CH, H, W, dtype=torch.float32)
    torch.onnx.export(
        model, dummy, str(model_path),
        input_names=['input'], output_names=['output'],
        opset_version=17, dynamic_axes=None, do_constant_folding=True,
        dynamo=False,
    )
    m = onnx.load(str(model_path))
    m.ir_version = 8
    onnx.checker.check_model(m)
    onnx.save(m, str(model_path))
    return model_path

def validate_onnx_model(model_path, task, scope='visible'):
    sess = ort.InferenceSession(str(model_path), providers=['CPUExecutionProvider'])
    splits = ['train','test'] if scope == 'visible' else ['train','test','arc-gen']
    right = total = wrongpix = 0
    first_wrong = None
    for split in splits:
        for idx, ex in enumerate(task.get(split, [])):
            if 'output' not in ex:
                continue
            pred = sess.run(['output'], {'input': grid_to_tensor(ex['input'])})[0]
            pred = (pred > 0.5).astype(np.float32)
            gold = expected_tensor(ex['output'])
            ok = np.array_equal(pred, gold)
            if ok:
                right += 1
            elif first_wrong is None:
                first_wrong = f'{split}:{idx}'
            wrongpix += int(np.sum(pred != gold))
            total += 1
    on = onnx.load(str(model_path))
    ops = {}
    for node in on.graph.node:
        ops[node.op_type] = ops.get(node.op_type, 0) + 1
    return {
        'right': right,
        'total': total,
        'wrongpix': wrongpix,
        'first_wrong': first_wrong,
        'size': Path(model_path).stat().st_size,
        'under_1_4mb': Path(model_path).stat().st_size < SIZE_LIMIT_BYTES,
        'forbidden': sorted(FORBIDDEN_OPS & set(ops)),
        'ops': ops,
    }


In [5]:
from pathlib import Path
import json
ROOT = Path.cwd()
OUT_DIR = ROOT / 'generated_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / f'{TASK_ID}.onnx'
MANIFEST_PATH = OUT_DIR / f'{TASK_ID}_manifest.json'

In [6]:
# Load task JSON. This cell includes a lightweight fallback loader so it works even if
# the large shared modelling cell above was not run before this cell.
from pathlib import Path
import json

def _candidate_task_roots():
    roots = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
        Path('/mnt/data'),
        Path('/mnt/data/generated_arc_notebooks_fixed'),
        Path('/mnt/data/generated_arc_notebooks/extracted'),
    ]
    seen = []
    for r in roots:
        if r not in seen:
            seen.append(r)
    return seen

if 'find_task_json' not in globals():
    def find_task_json(task_id):
        names = [f'{task_id}.json']
        for root in _candidate_task_roots():
            for name in names:
                p = root / name
                if p.exists():
                    return p
        raise FileNotFoundError(
            f'Cannot find {task_id}.json. Put the task JSON next to the notebook, '
            f'in the parent folder, or in /mnt/data.'
        )

if 'load_task' not in globals():
    def load_task(task_id):
        p = find_task_json(task_id)
        with open(p) as f:
            return json.load(f), p

task, task_path = load_task(TASK_ID)
print('task path:', task_path)
print('train examples:', len(task.get('train', [])))
print('test examples:', len(task.get('test', [])))
print('arc-gen examples:', len(task.get('arc-gen', [])))


task path: /kaggle/input/competitions/neurogolf-2026/task064.json
train examples: 4
test examples: 1
arc-gen examples: 262


In [7]:
# Small inspection: shape and color counts for the first training pair.
from collections import Counter
first = task['train'][0]
print('input shape:', (len(first['input']), len(first['input'][0])))
print('output shape:', (len(first['output']), len(first['output'][0])))
print('input colors:', Counter(v for row in first['input'] for v in row))
print('output colors:', Counter(v for row in first['output'] for v in row))

input shape: (9, 12)
output shape: (9, 12)
input colors: Counter({8: 94, 3: 12, 4: 2})
output colors: Counter({8: 90, 3: 12, 4: 6})


In [8]:
# Export plan for this task.
plan = {
    'task_id': TASK_ID,
    'builder': BUILDER,
    'family': FAMILY,
    'subtype': SUBTYPE,
    'uses_tree_methods': False,
    'uses_visible_template_lookup': False,
    'static_input_shape': [1, CH, H, W],
    'status_from_generation_run': 'passes',
}
print(json.dumps(plan, indent=2))

{
  "task_id": "task064",
  "builder": "RectangleConnector",
  "family": "arc_static_non_tree_symbolic",
  "subtype": "orthogonal connector from object rectangle to matching markers",
  "uses_tree_methods": false,
  "uses_visible_template_lookup": false,
  "static_input_shape": [
    1,
    10,
    30,
    30
  ],
  "status_from_generation_run": "passes"
}


In [9]:
# Task-specific model construction wrapper.
def build_current_model():
    return export_onnx_model(TASK_ID, MODEL_PATH)

def validate_current_model(scope='visible'):
    return validate_onnx_model(MODEL_PATH, task, scope=scope)

In [10]:
# Build ONNX model and enforce structural competition constraints.
build_current_model()
validation_report = validate_current_model('visible')
print(json.dumps(validation_report, indent=2))
assert validation_report['under_1_4mb'], validation_report
assert not validation_report['forbidden'], validation_report
if EXPECTED_VISIBLE_PASS:
    assert validation_report['right'] == validation_report['total'], validation_report
else:
    print('NOTE: this task is marked partial/WIP; visible correctness assertion is intentionally not enforced.')

/tmp/ipykernel_16/3602982844.py:553: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


{
  "right": 5,
  "total": 5,
  "wrongpix": 0,
  "first_wrong": null,
  "size": 70780,
  "under_1_4mb": true,
  "forbidden": [],
  "ops": {
    "Constant": 401,
    "ReduceSum": 4,
    "Greater": 45,
    "Cast": 69,
    "ArgMax": 2,
    "Equal": 20,
    "Concat": 2,
    "Mul": 113,
    "Sub": 22,
    "Slice": 54,
    "Add": 63,
    "CumSum": 44,
    "LessOrEqual": 4,
    "Clip": 23
  }
}


In [11]:
# Optional broader arc-gen check. This is informational because generated examples may include broader distributions.
optional_all_report = validate_current_model('all')
print(json.dumps(optional_all_report, indent=2))

{
  "right": 266,
  "total": 267,
  "wrongpix": 16,
  "first_wrong": "arc-gen:42",
  "size": 70780,
  "under_1_4mb": true,
  "forbidden": [],
  "ops": {
    "Constant": 401,
    "ReduceSum": 4,
    "Greater": 45,
    "Cast": 69,
    "ArgMax": 2,
    "Equal": 20,
    "Concat": 2,
    "Mul": 113,
    "Sub": 22,
    "Slice": 54,
    "Add": 63,
    "CumSum": 44,
    "LessOrEqual": 4,
    "Clip": 23
  }
}


In [12]:
# Model/version manifest.
run_manifest = {
    'task_id': TASK_ID,
    'model_version': MODEL_VERSION,
    'family': FAMILY,
    'subtype': SUBTYPE,
    'builder': BUILDER,
    'uses_tree_methods': False,
    'uses_visible_template_lookup': False,
    'visible_validation': validation_report,
    'arc_gen_validation': optional_all_report,
}
print(json.dumps(run_manifest, indent=2)[:2000])

{
  "task_id": "task064",
  "model_version": "task064-rectangle-connector",
  "family": "arc_static_non_tree_symbolic",
  "subtype": "orthogonal connector from object rectangle to matching markers",
  "builder": "RectangleConnector",
  "uses_tree_methods": false,
  "uses_visible_template_lookup": false,
  "visible_validation": {
    "right": 5,
    "total": 5,
    "wrongpix": 0,
    "first_wrong": null,
    "size": 70780,
    "under_1_4mb": true,
    "forbidden": [],
    "ops": {
      "Constant": 401,
      "ReduceSum": 4,
      "Greater": 45,
      "Cast": 69,
      "ArgMax": 2,
      "Equal": 20,
      "Concat": 2,
      "Mul": 113,
      "Sub": 22,
      "Slice": 54,
      "Add": 63,
      "CumSum": 44,
      "LessOrEqual": 4,
      "Clip": 23
    }
  },
  "arc_gen_validation": {
    "right": 266,
    "total": 267,
    "wrongpix": 16,
    "first_wrong": "arc-gen:42",
    "size": 70780,
    "under_1_4mb": true,
    "forbidden": [],
    "ops": {
      "Constant": 401,
      "ReduceSu

In [13]:
# Architecture report.
model = onnx.load(str(MODEL_PATH))
op_counts = {}
for node in model.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1
print('onnx_size_bytes:', MODEL_PATH.stat().st_size)
print('forbidden_ops:', sorted(FORBIDDEN_OPS & set(op_counts)))
print('op_counts:', op_counts)

onnx_size_bytes: 70780
forbidden_ops: []
op_counts: {'Constant': 401, 'ReduceSum': 4, 'Greater': 45, 'Cast': 69, 'ArgMax': 2, 'Equal': 20, 'Concat': 2, 'Mul': 113, 'Sub': 22, 'Slice': 54, 'Add': 63, 'CumSum': 44, 'LessOrEqual': 4, 'Clip': 23}


In [14]:
# Persist metadata next to ONNX.
with open(MANIFEST_PATH, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('manifest:', MANIFEST_PATH)

manifest: /kaggle/working/generated_models/task064_manifest.json


In [15]:
# Persist run metadata next to generated models.
summary_path = OUT_DIR / f'{TASK_ID}_verification_summary.json'
with open(summary_path, 'w') as f:
    json.dump({'visible': validation_report, 'all': optional_all_report}, f, indent=2)
print('summary:', summary_path)

summary: /kaggle/working/generated_models/task064_verification_summary.json


In [16]:
# Package single-task submission zip.
zip_path = ROOT / f'{TASK_ID}_submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(MODEL_PATH, MODEL_PATH.name)
print('submission zip:', zip_path)

submission_path=Path.cwd()/'submission.zip'
with zipfile.ZipFile(submission_path,'w',zipfile.ZIP_DEFLATED) as zf: 
    zf.write(MODEL_PATH, MODEL_PATH.name)

submission zip: /kaggle/working/task064_submission.zip
